In [ ]:

# ============================================================================================
# Cell 1: Environment Setup and Imports
# ============================================================================================
import os
import json
import logging
from pathlib import Path
from collections import defaultdict
import pandas as pd
from google.colab import drive

# Configure logging to track the merging process and catch missing or malformed files
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

mount_path = '/content/drive'
if not os.path.ismount(mount_path):
    try:
        drive.mount(mount_path)
    except ValueError:
        logging.warning("Mount point occupied by local files. Clearing for drive mount...")
        !rm -rf {mount_path}
        drive.mount(mount_path)
else:
    logging.info("Google Drive is already mounted and ready.")

In [ ]:

# ============================================================================================
# Cell 2: Schema Mapping Engine
# ============================================================================================
BASE_SCHEMA = {
    "image_key": "image_path",
    "label": "label",
    "pixel_error_point": "pixel_error_point",
    "pixel_error_grasp": "pixel_error_grasp",
    "success_rate": "success_rate",
    "prompt_sensitivity": "prompt_sensitivity"
}

SCHEMA_MAP = {
    "clip": {
        "image_key": "image_path",
        "label": "label",
        "pixel_error_point": "pixel_error",
        "pixel_error_grasp": "normalized_pixel_error",
        "success_rate": "success",
        "prompt_sensitivity": "prompt_sensitivity"
    },
    "grounding_dino": BASE_SCHEMA.copy(),
    "florence2": BASE_SCHEMA.copy()
}

# SCHEMA_MAP["clip"]["image_key"] = "filename"


In [ ]:

# ============================================================================================
# Cell 3: Robust Aggregation Class
# ============================================================================================
class ResultAggregator:
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.master_data = defaultdict(dict)

    def extract_image_name(self, path_or_name: str) -> str:
        return Path(path_or_name).name

    def load_and_normalize(self, model_name: str, filepath: str):
        file_path = Path(filepath)

        # Log the missing file but do not crash the script
        if not file_path.exists():
            logging.warning(f"[WORK IN PROGRESS] Cannot find data for {model_name}. It will be populated with empty values.")
            return

        try:
            with open(file_path, 'r') as f:
                data = json.load(f)

            logging.info(f"[SUCCESS] Loaded {len(data)} records for {model_name}.")
            schema = SCHEMA_MAP.get(model_name)

            for entry in data:
                raw_image_val = entry.get(schema["image_key"])
                if not raw_image_val:
                    continue

                image_name = self.extract_image_name(raw_image_val)

                # Register the ground truth label once
                if "label" not in self.master_data[image_name]:
                    self.master_data[image_name]["label"] = entry.get(schema["label"], "UNKNOWN")

                # Map specific metrics
                self.master_data[image_name][model_name] = {
                    "pixel_error_point": entry.get(schema["pixel_error_point"]),
                    "pixel_error_grasp": entry.get(schema["pixel_error_grasp"]),
                    "success_rate": entry.get(schema["success_rate"]),
                    "prompt_sensitivity": entry.get(schema["prompt_sensitivity"])
                }

        except json.JSONDecodeError:
            logging.error(f"[CORRUPTION] Malformed JSON file: {file_path}")
        except Exception as e:
            logging.error(f"[ERROR] Unexpected error loading {model_name}: {e}")

    def export_master_json(self, filename="master_comparative_results.json"):
        final_list = []

        empty_metrics = {
            "pixel_error_point": None,
            "pixel_error_grasp": None,
            "success_rate": None,
            "prompt_sensitivity": None
        }

        for image_name, data in self.master_data.items():
            record = {"image_name": image_name, "label": data.get("label", "UNKNOWN")}

            for model in ["clip", "grounding_dino", "florence2"]:
                record[model] = data.get(model, empty_metrics)

            final_list.append(record)

        output_path = self.output_dir / filename
        with open(output_path, 'w') as f:
            json.dump(final_list, f, indent=4)

        logging.info(f"[COMPLETE] Master JSON standardized and exported to: {output_path}")
        return final_list


In [ ]:

# ============================================================================================
# Cell 4: Execution Pipeline
# ============================================================================================
BASE_DIR = "/content/drive/MyDrive/eng521/Grasp Point Prediction/evaluation"

FILE_PATHS = {
    "clip": f"{BASE_DIR}/Clip_v3/clip_test_final_per_image.json",
    "grounding_dino": f"{BASE_DIR}/GroundingDINO/grounding_dino_test_final_per_image.json",
    "florence2": f"{BASE_DIR}/Florence2/florence2_eval_results.json"
}

SAVE_DIR = f"{BASE_DIR}/comparative_study_results"
aggregator = ResultAggregator(output_dir=SAVE_DIR)

for model_name, path in FILE_PATHS.items():
    aggregator.load_and_normalize(model_name, path)

master_data = aggregator.export_master_json()

df_master = pd.json_normalize(master_data)
print("\n[READY] Standardized Master DataFrame Preview:")
display(df_master.sample(5, random_state=42))


[READY] Standardized Master DataFrame Preview:


,image_name,label,clip.pixel_error_point,clip.pixel_error_grasp,clip.success_rate,clip.prompt_sensitivity,grounding_dino.pixel_error_point,grounding_dino.pixel_error_grasp,grounding_dino.success_rate,grounding_dino.prompt_sensitivity,florence2.pixel_error_point,florence2.pixel_error_grasp,florence2.success_rate,florence2.prompt_sensitivity
19,book_20.jpg,book,41.680852,0.028381,1,None,None,None,None,None,12.052508,11.470180,1,0.72
42,egg_8.jpg,egg,90.020363,0.131110,0,None,None,None,None,None,125.015023,31.860273,0,117.37
155,tape_19.jpg,tape,670.456231,0.193662,0,None,None,None,None,None,277.358297,676.114083,0,840.14
111,pen_1.jpg,pen,325.746703,0.147871,0,None,None,None,None,None,36.592743,216.958679,0,234.26
147,spoon_3.jpg,spoon,223.102821,0.189059,0,None,None,None,None,None,72.796983,72.410835,0,0.40


In [ ]:

# ============================================================================================
# Cell 5: Post-Execution Output Validation
# ============================================================================================
import os
import logging

EXPECTED_PATH = f"{SAVE_DIR}/master_comparative_results.json"

if os.path.exists(EXPECTED_PATH):
    logging.info(f"Validation successful. Master JSON located at: {EXPECTED_PATH}")
else:
    error_msg = (
        f"Validation failed. Master JSON not found at {EXPECTED_PATH}. "
        "Verify that the aggregation pipeline executed successfully and that "
        "directory naming conventions match the case-sensitive environment."
    )
    logging.error(error_msg)

    # raise FileNotFoundError(error_msg)